# **Arka Plan Kaldırma için GrabCut Algoritması**

- Bu derste arka plan kaldırma için GrabCut Algoritmasını kullanacağız

In [ ]:
import cv2
import dlib
import sys
import numpy as np
from matplotlib import pyplot as plt


def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()

### **Grabcut Nasıl Çalışır?**

- Kullanıcı dikdörtgeni belirler. Bu dikdörtgenin dışındaki her şey arka plan olarak alınacaktır. Dikdörtgenin içindeki her şey bilinmiyor. 
- Algoritma ön plan ve arka plan piksellerini etiketler (veya sabit etiketler)
- Daha sonra ön plan ve arka planı modellemek için bir Gauss Karışım Modeli (GMM) kullanılır.
- Verdiğimiz verilere bağlı olarak, GMM öğrenir ve yeni piksel dağılımı oluşturur. Yani, **bilinmeyen pikseller, renk istatistikleri açısından diğer zor etiketlenmiş piksellerle olan ilişkisine bağlı olarak olası ön plan veya olası arka plan** olarak etiketlenir (Tıpkı kümeleme gibi).
- Bu piksel dağılımından bir grafik oluşturulur. Graflardaki düğümler piksellerdir. Ek olarak iki düğüm eklenir, Kaynak düğümü ve Sink düğümü. Her ön plan pikseli Source düğümüne ve her arka plan pikseli Sink düğümüne bağlanır.
- Pikselleri kaynak düğüme/son düğüme bağlayan kenarların ağırlıkları, bir pikselin ön plan/arka plan olma olasılığı ile tanımlanır. Pikseller arasındaki ağırlıklar kenar bilgisi veya piksel benzerliği ile tanımlanır. Piksel renginde büyük bir fark varsa, aralarındaki kenar düşük bir ağırlık alacaktır.
= Daha sonra grafiği bölümlere ayırmak için bir mincut algoritması kullanılır. Grafiği minimum maliyet fonksiyonu ile kaynak düğüm ve sink düğüm olarak ikiye ayırır. Maliyet fonksiyonu, kesilen kenarların tüm ağırlıklarının toplamıdır. Kesmeden sonra, Kaynak düğümüne bağlı tüm pikseller ön plan haline gelir ve Sink düğümüne bağlı olanlar arka plan haline gelir.
- Sınıflandırma yakınsayana kadar işleme devam edilir.

![](https://docs.opencv.org/3.4/grabcut_scheme.jpg)

Paper - http://dl.acm.org/citation.cfm?id=1015720

Learn more - https://docs.opencv.org/3.4/d8/d83/tutorial_py_grabcut.html

In [ ]:
# Resmimizi yükleyelim
image = cv2.imread('../files/images/woman.jpeg')
copy = image.copy()
# Orijinal görüntümüzle aynı boyutta (genişlik, yükseklik) bir maske (sıfırlardan oluşan uint8 veri tipi) oluşturalım
mask = np.zeros(image.shape[:2], np.uint8)

bgdModel = np.zeros((1,65), np.float64)
fgdModel = np.zeros((1,65), np.float64)

# ROI(Region of İnterest) İligli Bölge Manuel olarak ayarlanmalı veya cv2.selectROI() ile seçilmelidir
x1, y1, x2, y2 = 190, 70, 350, 310
start = (x1, y1)
end = (x2, y2)

# Format is X,Y,W,H
rect = (x1,y1,x2-x1,y2-y1)

# Show Rectangle
cv2.rectangle(copy, start, end, (0,0,255), 3)
imshow("Input Image", copy)

#### **Grabcut Parametreleri**

cv2.grabCut(image, mask, rect, bgdModel, fgdModel, iterCount, mode)

- image: İşlenecek giriş resmi (numpy array, genelde BGR formatında).
- mask: Resim ile aynı boyutta, tek kanallı (uint8) bir maske.
  Algoritma çalıştıktan sonra her pikselin arka plan mı ön plan mı olduğunu bu maskede göreceğiz.
  Maske değerleri:
    - cv2.GC_BGD = 0 → kesin arka plan
    - cv2.GC_FGD = 1 → kesin ön plan
    - cv2.GC_PR_BGD = 2 → muhtemel arka plan
    - cv2.GC_PR_FGD = 3 → muhtemel ön plan
- rect: (x, y, w, h) şeklinde bir dikdörtgen. Bu dikdörtgenin içinde kalan nesneyi ön plan kabul eder, dışını ise arka plan varsayar.
- bgdModel, fgdModel
  Algoritmanın kullandığı geçici veri yapıları. Başlangıçta boş array olarak tanımlanır, algoritma kendi doldurur
- iterCount: Algoritmanın çalıştırılacağı iterasyon (tekrar) sayısı. Daha fazla tekrar = daha iyi sonuç ama daha yavaş.
- cv2.GC_INIT_WITH_RECT (mode): Başlatma yöntemi. Burada başlangıçta dikdörtgen (rect) ile ön plan belirleniyor.
Alternatif: cv2.GC_INIT_WITH_MASK → maske üzerinde elle ön/arka plan pikselleri işaretlenerek başlanabilir

In [ ]:
# Algoritma 5 iterasyon boyunca çalışacaktır. Dikdörtgen kullandığımız için mod cv.GC_INIT_WITH_RECT olmalıdır. 
# Grabcut maske görüntüsünü değiştirir. 

cv2.grabCut(image, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)

# Bu satırdan sonra mask güncellenir ve her piksele 0, 1, 2 veya 3 değeri atanır:
# 0 → kesin arka plan
# 1 → kesin ön plan
# 2 → muhtemel arka plan
# 3 → muhtemel ön plan

mask2 = np.where((mask==2)|(mask==0),0,1).astype('uint8')
# np.where ile yeni bir maske oluşturuluyor:
# Eğer mask değeri 2 (muhtemel arka plan) veya 0 (kesin arka plan) ise → 0 yap
# Aksi halde (1 veya 3, yani ön plan) → 1 yap
# Sonuçta elimizde sadece 0 = arka plan, 1 = ön plan olacak ikili (binary) bir maske oluyor.

image = image * mask2[:,:,np.newaxis]
# mask2 tek kanallı (gri) maske, [:,:,np.newaxis] ile 3 kanallı hale getiriliyor.
# çarparak resmin sadece ön planı görünür hale gelir, arka plan silinir.

imshow("Mask", mask * 80)
imshow("Mask2", mask2 * 255)
imshow("Image", image) # Sonuç: sadece ön plan kalmış, arka plan siyah yapılmış hali.